# AHR v2 — fresh 9×9 foundation and recovery

Purpose: define the fresh experiment, import only exported definitions, and prove interruption-safe checkpoint recovery before costly learning. Inputs: this workspace and the new rules/network/search notebooks. Outputs: source identities, environment metadata, a bounded recovery proof and disposable checkpoints. Expected runtime: under one minute on CPU. Checkpoints: `checkpoints/ahr-v2-recovery-<id>/`; fixture checkpoints are validation artifacts, never player candidates.

The first milestone is a playable 9×9 foundation: exact area rules, Scout move energies, Worker outcome energies and batched PUCT with visit-count policy targets. Model weights are fresh. A fixed batch scheduler supplies the initial Queen role. Learned Queen allocation and Lorentzian workspace refinement are later, separately measured components; neither is claimed to help yet.

The previous v5 run stayed in tactical warmup. This restart will make complete-game learning the main path. Tactical tasks will supplement that path; their old 95–99% threshold will not be a prerequisite for starting full-game learning. Truncated games remain UNKNOWN and supply no invented terminal targets. Final data stays separate.

Training-source preference is pending the user's answer: KataGo labels or AHR self-play. This foundation and its short recovery tests need neither external labels nor an expensive training run. Later long runs require a fresh measured throughput budget and exact resume proof, use detached tmux when over 30 minutes, and preserve model/optimizer/scheduler/scaler/all used RNGs/data and active self-play/search state. Store at stage boundaries and within 250 updates or 15 minutes; retain latest two valid states plus best. One writer per run.

Exact fixture resume: load this notebook's tagged definitions in a fresh project kernel and call `go2_optimizer_fixture(PROJECT, saved_run_id, total_steps=8)`. Identity, manifest and payload corruption reject recovery; no silent restart. Storage is the existing persistent boot disk; deletion policy remains unverified and off-instance backup remains deferred by the user.

Design references: [AlphaZero](https://arxiv.org/abs/1712.01815), [KataGo explicit rules](https://lightvector.github.io/KataGo/rules.html), [KataGo analysis outputs](https://github.com/lightvector/KataGo/blob/master/docs/Analysis_Engine.md). These are methodological references, not performance guarantees.

## Exported contracts and checkpoint storage

In [ ]:
from pathlib import Path
import contextlib, copy, dataclasses, datetime as dt, fcntl, hashlib, io, json, os, random, sys, time, uuid
import nbformat, numpy as np, torch

GO2_ROOT='experiments/ahr_v2'
GO2_SCHEMA='ahr-go-v2-fresh'
GO2_NOTEBOOKS=('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb')
_GO2_HELD_LOCKS=set()
GO2_REQUIRED_STATE=frozenset(('schema','identity','model','optimizer','scheduler','scaler','rng','sampler','global_step','config','metrics'))

def go2_project(path=None):
    start=Path(path or Path.cwd()).resolve()
    for p in (start,*start.parents):
        if (p/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():return p
    raise RuntimeError('Run in the AHR Go workspace')

def go2_json_default(value):
    if isinstance(value,Path):return str(value)
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,np.ndarray):return value.tolist()
    if dataclasses.is_dataclass(value):return dataclasses.asdict(value)
    if isinstance(value,(set,frozenset)):return sorted(value)
    raise TypeError('Not an identity/JSON value: '+type(value).__name__)

def go2_digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False,default=go2_json_default).encode()).hexdigest()

def go2_code_sha(path):
    cells=[c.source for c in nbformat.read(Path(path),4).cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])]
    if not cells:raise RuntimeError('No explicit ahr-export definitions: '+str(path))
    return go2_digest(cells)

def go2_import_exports(path,namespace,expected_sha=None):
    path=Path(path);digest=go2_code_sha(path)
    if expected_sha is not None and digest!=expected_sha:raise RuntimeError('Exported source identity changed: '+str(path))
    for index,cell in enumerate(nbformat.read(path,4).cells):
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,str(path)+':ahr-export:'+str(index),'exec'),namespace)
    return digest

def go2_source_identity(project,require_all=True):
    result={}
    for name in GO2_NOTEBOOKS:
        p=Path(project)/GO2_ROOT/name
        if not p.is_file():
            if require_all:raise FileNotFoundError(p)
            continue
        result[str(p.relative_to(project))]=go2_code_sha(p)
    return result

def go2_file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def go2_tensor_digest(state_dict):
    h=hashlib.sha256()
    for key,tensor in sorted(state_dict.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(json.dumps([key,str(t.dtype),list(t.shape)],separators=(',',':')).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

def go2_now():return dt.datetime.now(dt.timezone.utc).isoformat()

def go2_atomic_bytes(path,payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name('.'+path.name+'.'+uuid.uuid4().hex+'.tmp')
    with temp.open('xb') as stream:stream.write(payload);stream.flush();os.fsync(stream.fileno())
    os.replace(temp,path)
    fd=os.open(str(path.parent),os.O_DIRECTORY)
    try:os.fsync(fd)
    finally:os.close(fd)

def go2_atomic_json(path,value):
    go2_atomic_bytes(path,(json.dumps(value,sort_keys=True,indent=2,allow_nan=False,default=go2_json_default)+'\n').encode())

def go2_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)

def go2_capture_rng(cuda_devices=(),generators=None):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state().clone(),
        'torch_cuda':{str(int(i)):torch.cuda.get_rng_state(int(i)).cpu().clone() for i in cuda_devices},
        'torch_generators':{key:gen.get_state().clone() for key,gen in (generators or {}).items()}}

def go2_restore_rng(rng,generators=None):
    generators=generators or {}
    if set(generators)!=set(rng['torch_generators']):raise RuntimeError('Private RNG generators differ')
    random.setstate(rng['python']);np.random.set_state(rng['numpy']);torch.set_rng_state(rng['torch_cpu'])
    for key,value in rng['torch_cuda'].items():
        if not torch.cuda.is_available() or int(key)>=torch.cuda.device_count():raise RuntimeError('Saved CUDA RNG device unavailable')
        torch.cuda.set_rng_state(value,int(key))
    for key,value in rng['torch_generators'].items():generators[key].set_state(value)

def go2_nested_equal(a,b):
    if isinstance(a,torch.Tensor):return isinstance(b,torch.Tensor) and a.dtype==b.dtype and a.shape==b.shape and torch.equal(a,b)
    if isinstance(a,np.ndarray):return isinstance(b,np.ndarray) and a.dtype==b.dtype and np.array_equal(a,b)
    if isinstance(a,dict):return isinstance(b,dict) and a.keys()==b.keys() and all(go2_nested_equal(a[k],b[k]) for k in a)
    if isinstance(a,(list,tuple)):return type(a) is type(b) and len(a)==len(b) and all(go2_nested_equal(x,y) for x,y in zip(a,b))
    return type(a) is type(b) and a==b

@contextlib.contextmanager
def go2_checkpoint_lock(run_dir):
    run_dir=Path(run_dir).resolve();run_dir.mkdir(parents=True,exist_ok=True)
    if run_dir in _GO2_HELD_LOCKS:raise RuntimeError('Another writer already owns this run in this process')
    with (run_dir/'.checkpoint.lock').open('a+b') as stream:
        try:fcntl.flock(stream.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError as e:raise RuntimeError('Another process owns this run: '+str(run_dir)) from e
        _GO2_HELD_LOCKS.add(run_dir)
        try:yield run_dir
        finally:_GO2_HELD_LOCKS.remove(run_dir);fcntl.flock(stream.fileno(),fcntl.LOCK_UN)

def go2_manifest_states(run_dir):
    run_dir=Path(run_dir);path=run_dir/'manifest.json'
    if not path.exists():
        if list(run_dir.glob('*.pt')):raise RuntimeError('Checkpoint payload without a manifest; no restart permitted')
        return None,{}
    try:manifest=json.loads(path.read_text())
    except (ValueError,OSError) as e:raise RuntimeError('Invalid checkpoint manifest; no restart permitted') from e
    entries=manifest.get('entries',[])
    if manifest.get('schema')!='ahr-go-v2-checkpoint-manifest-v1' or not entries or len({e.get('file') for e in entries})!=len(entries):raise RuntimeError('Malformed checkpoint manifest')
    if manifest.get('latest')!=max(entries,key=lambda e:e['global_step'])['file'] or manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Invalid latest/best checkpoint cursor')
    states={}
    for entry in entries:
        if Path(entry['file']).name!=entry['file']:raise RuntimeError('Checkpoint filename must be local')
        payload=run_dir/entry['file']
        if not payload.is_file() or go2_file_sha(payload)!=entry['sha256']:raise RuntimeError('Corrupt retained checkpoint; no fallback permitted')
        try:state=torch.load(payload,map_location='cpu',weights_only=False)
        except Exception as e:raise RuntimeError('Unreadable checkpoint; no restart permitted') from e
        if not isinstance(state,dict) or GO2_REQUIRED_STATE-set(state) or state['schema']!=GO2_SCHEMA or type(state['global_step']) is not int or state['global_step']!=entry['global_step']:raise RuntimeError('Incomplete/wrong-schema checkpoint')
        states[entry['file']]=state
    if len({go2_digest(s['identity']) for s in states.values()})!=1:raise RuntimeError('Mixed checkpoint identities')
    return manifest,states

def go2_load_checkpoint(run_dir,identity=None,which='latest',restore_rng=False,generators=None):
    if which not in ('latest','best'):raise ValueError('Choose latest or best')
    manifest,states=go2_manifest_states(run_dir)
    if manifest is None:return None,None
    if identity is not None and any(go2_digest(s['identity'])!=go2_digest(identity) for s in states.values()):raise RuntimeError('Checkpoint source/configuration/data identity differs; no restart permitted')
    name=manifest[which];state=states[name];entry=copy.deepcopy(next(e for e in manifest['entries'] if e['file']==name))
    entry['uncommitted_payloads']=sorted(p.name for p in Path(run_dir).glob('*.pt') if p.name not in states)
    if restore_rng:go2_restore_rng(state['rng'],generators)
    return state,entry

def go2_save_checkpoint(run_dir,state,best=False,resume_cell='00_contracts_and_recovery.ipynb: exact saved run/config; tagged exports only'):
    run_dir=Path(run_dir).resolve()
    if run_dir not in _GO2_HELD_LOCKS:raise RuntimeError('Checkpoint writes require the held single-writer lock')
    if GO2_REQUIRED_STATE-set(state) or state['schema']!=GO2_SCHEMA or type(state['global_step']) is not int or state['global_step']<0:raise ValueError('Full typed checkpoint state required')
    manifest,states=go2_manifest_states(run_dir)
    if manifest is not None:
        if state['global_step']<=max(s['global_step'] for s in states.values()):raise RuntimeError('Checkpoint step must increase')
        if any(go2_digest(s['identity'])!=go2_digest(state['identity']) for s in states.values()):raise RuntimeError('Cannot mix checkpoint identities')
    entries=copy.deepcopy(manifest['entries']) if manifest else []
    buffer=io.BytesIO();torch.save(state,buffer)
    name='step-'+str(state['global_step']).zfill(9)+'-'+uuid.uuid4().hex[:8]+'.pt';go2_atomic_bytes(run_dir/name,buffer.getvalue())
    entry={'file':name,'global_step':state['global_step'],'sha256':go2_file_sha(run_dir/name)};entries.append(entry)
    best_name=name if best or manifest is None else manifest['best']
    keep={e['file'] for e in sorted(entries,key=lambda e:e['global_step'])[-2:]}|{best_name}
    updated={'schema':'ahr-go-v2-checkpoint-manifest-v1','entries':[e for e in entries if e['file'] in keep],'latest':name,'best':best_name,'resume_cell':resume_cell,'updated_at_utc':go2_now()}
    go2_atomic_json(run_dir/'manifest.json',updated)
    for old in entries:
        if old['file'] not in keep:(run_dir/old['file']).unlink(missing_ok=True)
    return entry


## A real optimizer and fresh-kernel resume proof

In [ ]:
def go2_optimizer_fixture(project,run_id,total_steps=8,stop_after=None):
    project=Path(project);directory=project/'checkpoints'/run_id
    config={'purpose':'disposable_optimizer_recovery_validation','seed':61241,'device':'cpu','total_steps':int(total_steps)}
    identity={'schema':GO2_SCHEMA,'config':config,'sources':{GO2_ROOT+'/'+GO2_NOTEBOOKS[0]:go2_code_sha(project/GO2_ROOT/GO2_NOTEBOOKS[0])},'torch':torch.__version__}
    with go2_checkpoint_lock(directory):
        saved,_=go2_load_checkpoint(directory,identity)
        go2_seed(config['seed']);torch.set_num_threads(2)
        model=torch.nn.Linear(4,2);optimizer=torch.optim.AdamW(model.parameters(),lr=.003)
        scheduler=torch.optim.lr_scheduler.ConstantLR(optimizer,factor=1.0,total_iters=total_steps)
        scaler=torch.amp.GradScaler('cuda',enabled=False);generator=torch.Generator().manual_seed(8226)
        state={'schema':GO2_SCHEMA,'identity':identity,'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},'sampler':{'position':0},'global_step':0,'config':config,'metrics':{'losses':[]},'replay':[],'self_play':None,'search':None}
        if saved is not None:
            state=saved;model.load_state_dict(saved['model']);optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler']);scaler.load_state_dict(saved['scaler']);go2_restore_rng(saved['rng'],{'data':generator})
        boundary=min(total_steps,state['global_step']+stop_after) if stop_after is not None else total_steps
        while state['global_step']<boundary:
            x=torch.randn(5,4,generator=generator);target=torch.randn(5,2)+float(np.random.uniform())+random.random()
            optimizer.zero_grad(set_to_none=True);loss=torch.nn.functional.mse_loss(model(x),target);loss.backward();optimizer.step();scheduler.step()
            state['global_step']+=1;state['sampler']['position']+=len(x);state['metrics']['losses'].append(float(loss.detach()))
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),rng=go2_capture_rng(generators={'data':generator}))
            go2_save_checkpoint(directory,state,best=state['global_step']==1,resume_cell='00: go2_optimizer_fixture(PROJECT, saved_run_id, total_steps=8)')
        return state

def go2_validate_recovery(project):
    import subprocess
    project=Path(project);token=uuid.uuid4().hex[:10];full='ahr-v2-recovery-'+token+'-full';resumed='ahr-v2-recovery-'+token+'-resumed'
    baseline=go2_optimizer_fixture(project,full);partial=go2_optimizer_fixture(project,resumed,stop_after=4)
    proof_dir=project/'reports/ahr_v2/fixtures';proof_dir.mkdir(parents=True,exist_ok=True)
    book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}})
    setup="from pathlib import Path\nimport nbformat\nPROJECT=Path("+repr(str(project))+ ")\nfor c in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:\n    if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[]):exec(c.source,globals())\n"
    book.cells=[nbformat.v4.new_code_cell(setup),nbformat.v4.new_code_cell('state=go2_optimizer_fixture(PROJECT,'+repr(resumed)+',total_steps=8)\nassert state["global_step"]==8')]
    path=proof_dir/('optimizer-resume-'+token+'.ipynb');nbformat.write(book,path)
    subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=120',str(path)],check=True,capture_output=True,text=True)
    after,entry=go2_load_checkpoint(project/'checkpoints'/resumed,baseline['identity'])
    assert go2_nested_equal(baseline,after),'Fresh-kernel model/optimizer/RNG/sampler/metrics differ'
    run=project/'checkpoints'/resumed;manifest=json.loads((run/'manifest.json').read_text());checks={'genuine_optimizer_steps':True,'fresh_kernel_exact_full_state':True,'latest_two_plus_best':len(manifest['entries'])==3,'private_and_global_rng_restored':True}
    payload=run/manifest['latest'];original=payload.read_bytes();payload.write_bytes(original+b'invalid')
    try:
        try:go2_load_checkpoint(run,baseline['identity'])
        except RuntimeError:checks['corruption_rejected']=True
        else:raise AssertionError('Corruption was silently accepted')
    finally:payload.write_bytes(original)
    changed=copy.deepcopy(baseline['identity']);changed['config']['seed']+=1
    try:go2_load_checkpoint(run,changed)
    except RuntimeError:checks['changed_identity_rejected']=True
    else:raise AssertionError('Different source/configuration accepted')
    with go2_checkpoint_lock(run):
        try:
            with go2_checkpoint_lock(run):pass
        except RuntimeError:checks['second_writer_rejected']=True
        else:raise AssertionError('Concurrent writer accepted')
    receipt={'status':'PASS','checks':checks,'optimizer_updates_per_path':8,'full_run':full,'resumed_run':resumed,'resume_notebook':str(path.relative_to(project)),'sources':go2_source_identity(project,require_all=False),'scope':'Checkpoint-engine fixture only; not an AHR training result','validated_at_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/checkpoint_validation.json',receipt)
    return receipt


In [ ]:
PROJECT=go2_project()
result=go2_validate_recovery(PROJECT)
from IPython.display import display, Markdown
display(Markdown('| Recovery check | Result |\n|---|---|\n'+'\n'.join('| '+k.replace('_',' ')+' | '+('PASS' if v else 'FAIL')+' |' for k,v in result['checks'].items())))
print('Disposable optimizer recovery: '+result['status']+'; no player-strength claim.')

## Notebook roadmap

| Notebook | This foundation |
|---|---|
| 00 | Source contracts, reproducible environment and full-state checkpoints |
| 01 | Exact legal moves, scoring, history, independent verification and replay |
| 02 | Fresh Go encoder, Scout/Worker energy heads and typed learning targets |
| 03 | Batched PUCT, visit-count targets and resumed active search |

Next: complete-game self-play and replay/training, source-specific teacher data if selected, independent GNU Go development/final arena, and a play/watch interface. Add recurrent workspace, Lorentzian coordinates and learned Transformer Queen as measured comparisons after the base player works. Strong 9×9 play would not establish 19×19 strength.